In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra|
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df = pd.read_csv('/kaggle/input/stacksample/Tags.csv')
tag = []
# print(len(df['Tag']))
# print(len(df['Id']))
for i in range(0, len(df)):
    if df['Tag'][i] == 'python':
        tag.append(df['Id'][i])
        
print(len(tag))

In [ ]:
df = pd.read_csv('/kaggle/input/stacksample/Questions.csv', encoding = 'ISO-8859-1')
questions = []
answers = []
print(len(df['Id'].unique()))
for i in range(0, 100000):
    if df['Id'][i] in tag:
        questions.append(df['OwnerUserId'][i])
        questions.append(df['Body'][i])
        
print(len(questions))

In [ ]:
questions[:10]

In [ ]:
df = pd.read_csv('/kaggle/input/stacksample/Answers.csv', encoding = 'ISO-8859-1')
answers = []
print(len(df['Id'].unique()))
for i in range(0, 300000):
    if df['Id'][i] in tag:
        answers.append(df['ParentId'][i])
        answers.append(df['Body'][i])

In [ ]:
answers[:10]

***Human Conversation dialogs***

In [ ]:
dialogs = []
with open('/kaggle/input/human-conversation-training-data/human_chat.txt', 'r') as w:
    for line in w:
        try:
            dialogs.append(line.split(': ')[1].split("\n")[0])
        except:
            print('done')
        
len(dialogs)
dialogs

In [ ]:
len(dialogs)

***Mental Health Question Answers***

In [ ]:
mh = pd.read_csv('/kaggle/input/mental-health-faq-for-chatbot/Mental_Health_FAQ.csv')
mh.head

In [ ]:
data = list(zip(mh.Questions, mh.Answers))
qna = []
for i in data:
    data1 = []
    data1.append(i[0])
    data1.append(i[1])
    qna.append(data1)
    
print(list(mh.Answers))

***Empathetic Dialogues***

In [ ]:
ed = pd.read_csv('/kaggle/input/empathetic-dialogues-facebook-ai/emotion-emotion_69k.csv')
ed.head

In [ ]:
ed.columns

In [ ]:
emapthetic_dialogs = []
for i in ed.empathetic_dialogues:
    try: 
        emapthetic_dialogs.append(i.split('\nAgent :')[0].split('Customer :')[1])
    except:
        continue
    
len(emapthetic_dialogs)
emapthetic_dialogs

***Topical Chat***

In [ ]:
tc = pd.read_csv('/kaggle/input/chatbot-dataset-topical-chat/topical_chat.csv')
tc.head

In [ ]:
tc_dialogs = list(tc.message)
tc_dialogs[:50]
list(tc.message)

In [ ]:
len(tc_dialogs)

***Ask Reddit Question and Answers***

In [ ]:
# ar = pd.read_csv('/kaggle/input/askreddit-questions-and-answers/reddit_questions.csv')

***Jeopardy questions***

In [ ]:
jq = pd.read_csv('/kaggle/input/200000-jeopardy-questions/JEOPARDY_CSV.csv')
jq.head

In [ ]:
jq.describe

In [ ]:
jq.columns

In [ ]:
jq[' Question']

In [ ]:
jq[' Answer']

In [ ]:
data = list(zip(jq[' Question'], jq[' Answer']))
jqna = []
for i in data:
    data1 = []
    data1.append(str(i[0]))
    data1.append(str(i[1]))
    jqna.append(data1)
    
len(jqna)

***150k PYTHON SOURCE CODE DATASET***

In [ ]:
import tarfile
tar = tarfile.open("/kaggle/input/150k-python-dataset/py150.tar_1")
tar.extractall()
tar.close()

In [ ]:
ls

In [ ]:
import sys
import json as json
import ast

def PrintUsage():
    sys.stderr.write("""
Usage:
    parse_python.py <file>

""")
    exit(1)

def read_file_to_string(filename):
    f = open(filename, 'rt')
    s = f.read()
    f.close()
    return s

def parse_file(filename):
    global c, d
    tree = ast.parse(read_file_to_string(filename), filename)
    
    json_tree = []
    def gen_identifier(identifier, node_type = 'identifier'):
        pos = len(json_tree)
        json_node = {}
        json_tree.append(json_node)
        json_node['type'] = node_type
        json_node['value'] = identifier
        return pos
    
    def traverse_list(l, node_type = 'list'):
        pos = len(json_tree)
        json_node = {}
        json_tree.append(json_node)
        json_node['type'] = node_type
        children = []
        for item in l:
            children.append(traverse(item))
        if (len(children) != 0):
            json_node['children'] = children
        return pos
        
    def traverse(node):
        pos = len(json_tree)
        json_node = {}
        json_tree.append(json_node)
        json_node['type'] = type(node).__name__
        children = []
        if isinstance(node, ast.Name):
            json_node['value'] = node.id
        elif isinstance(node, ast.Num):
            json_node['value'] = unicode(node.n)
        elif isinstance(node, ast.Str):
            json_node['value'] = node.s.decode('utf-8')
        elif isinstance(node, ast.alias):
            json_node['value'] = unicode(node.name)
            if node.asname:
                children.append(gen_identifier(node.asname))
        elif isinstance(node, ast.FunctionDef):
            json_node['value'] = unicode(node.name)
        elif isinstance(node, ast.ClassDef):
            json_node['value'] = unicode(node.name)
        elif isinstance(node, ast.ImportFrom):
            if node.module:
                json_node['value'] = unicode(node.module)
        elif isinstance(node, ast.Global):
            for n in node.names:
                children.append(gen_identifier(n))
        elif isinstance(node, ast.keyword):
            json_node['value'] = unicode(node.arg)
        

        # Process children.
        if isinstance(node, ast.For):
            children.append(traverse(node.target))
            children.append(traverse(node.iter))
            children.append(traverse_list(node.body, 'body'))
            if node.orelse:
                children.append(traverse_list(node.orelse, 'orelse'))
        elif isinstance(node, ast.If) or isinstance(node, ast.While):
            children.append(traverse(node.test))
            children.append(traverse_list(node.body, 'body'))
            if node.orelse:
                children.append(traverse_list(node.orelse, 'orelse'))
        elif isinstance(node, ast.With):
            children.append(traverse(node.context_expr))
            if node.optional_vars:
                children.append(traverse(node.optional_vars))
            children.append(traverse_list(node.body, 'body'))
        elif isinstance(node, ast.TryExcept):
            children.append(traverse_list(node.body, 'body'))
            children.append(traverse_list(node.handlers, 'handlers'))
            if node.orelse:
                children.append(traverse_list(node.orelse, 'orelse'))
        elif isinstance(node, ast.TryFinally):
            children.append(traverse_list(node.body, 'body'))
            children.append(traverse_list(node.finalbody, 'finalbody'))
        elif isinstance(node, ast.arguments):
            children.append(traverse_list(node.args, 'args'))
            children.append(traverse_list(node.defaults, 'defaults'))
            if node.vararg:
                children.append(gen_identifier(node.vararg, 'vararg'))
            if node.kwarg:
                children.append(gen_identifier(node.kwarg, 'kwarg'))
        elif isinstance(node, ast.ExceptHandler):
            if node.type:
                children.append(traverse_list([node.type], 'type'))
            if node.name:
                children.append(traverse_list([node.name], 'name'))
            children.append(traverse_list(node.body, 'body'))
        elif isinstance(node, ast.ClassDef):
            children.append(traverse_list(node.bases, 'bases'))
            children.append(traverse_list(node.body, 'body'))
            children.append(traverse_list(node.decorator_list, 'decorator_list'))
        elif isinstance(node, ast.FunctionDef):
            children.append(traverse(node.args))
            children.append(traverse_list(node.body, 'body'))
            children.append(traverse_list(node.decorator_list, 'decorator_list'))
        else:
            # Default handling: iterate over children.
            for child in ast.iter_child_nodes(node):
                if isinstance(child, ast.expr_context) or isinstance(child, ast.operator) or isinstance(child, ast.boolop) or isinstance(child, ast.unaryop) or isinstance(child, ast.cmpop):
                    # Directly include expr_context, and operators into the type instead of creating a child.
                    json_node['type'] = json_node['type'] + type(child).__name__
                else:
                    children.append(traverse(child))
                
        if isinstance(node, ast.Attribute):
            children.append(gen_identifier(node.attr, 'attr'))
                
        if (len(children) != 0):
            json_node['children'] = children
        return pos
    
    traverse(tree)
    return json.dumps(json_tree, separators=(',', ':'), ensure_ascii=False)

if __name__ == "__main__":
    if len(sys.argv) != 2:
        PrintUsage()
    try:
        print('file')
        #print(parse_file(sys.argv[1]))
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass

***Python Questions Dataset***

In [ ]:
import pandas as pd
import numpy as np
from nltk.corpus import stopwords
import re
from wordcloud import WordCloud, STOPWORDS 
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from transformers import (GPT2Config,GPT2LMHeadModel,GPT2Tokenizer)
import torch
from string import punctuation as pnc
from collections import Counter
from scipy import spatial
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
import torch
import pylab as pl
pd.set_option('display.max_colwidth', -1)

In [ ]:
import nltk
nltk.download('stopwords')

In [ ]:
questions = pd.read_csv("/kaggle/input/pythonquestions/Questions.csv", encoding = "ISO-8859-1")
print(len(questions))
display(questions.head(5))

In [ ]:
print("Number of unique Questions : ", questions['Id'].nunique())

In [ ]:
questions['qLen'] = questions['Title'].apply(lambda x : len(x.split(" ")))
questions['qBodyLen'] = questions['Body'].apply(lambda x : len(x.split(" ")))

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
questions['qLen'].hist(bins=35)
plt.title("No. of words in Title")

In [ ]:
questions[questions['qBodyLen']<500]['qBodyLen'].hist(bins=100)
plt.title("No. of words in Body")

In [ ]:
def getWordCloud(df,col):
  comment_words = '' 
  stopwords = set(STOPWORDS) 
    
  for val in tqdm(df[col]): 
        
      val = str(val) 
      tokens = val.split() 
        
      for i in range(len(tokens)): 
          tokens[i] = tokens[i].lower() 
        
      comment_words += " ".join(tokens)+" "
    
  wordcloud = WordCloud(width = 800, height = 800, 
                  background_color ='white', 
                  stopwords = stopwords, 
                  min_font_size = 10).generate(comment_words) 
    
                       
  plt.figure(figsize = (5, 5), facecolor = None) 
  plt.imshow(wordcloud) 
  plt.axis("off")
  plt.tight_layout(pad = 0) 
    
  plt.show()

In [ ]:
getWordCloud(questions,'Title')

In [ ]:
stop = stopwords.words('english')
def preprocess(df, col):
  df['preprocessed'+col] = df[col].apply(lambda x : " ".join([word for word in x.split(" ") if word not in stop]))
  df['preprocessed'+col] = df['preprocessed'+col].str.replace('[^a-zA-Z0-9 ]', '')
  df['preprocessed'+col] = df['preprocessed'+col].str.lower()
  return df

In [ ]:
questions = preprocess(questions, 'Title')

In [ ]:
tags = pd.read_csv("/kaggle/input/pythonquestions/Tags.csv", encoding = "ISO-8859-1")
print(len(tags))
display(tags.head(5))

In [ ]:
print("Number of unique Tags : ", tags['Tag'].nunique())

In [ ]:
fig, ax = plt.subplots()
tags[tags['Tag']!='python']['Tag'].value_counts().sort_values(ascending = False)[:20].plot(ax=ax, kind='bar')

In [ ]:
config_class, model_class, tokenizer_class = GPT2Config, GPT2LMHeadModel, GPT2Tokenizer
model = model_class.from_pretrained('gpt2')
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

In [ ]:
preprocessedTitle = questions['preprocessedTitle'].values
QID = questions['Id'].values
print(len(preprocessedTitle), len(QID))

In [ ]:
encodedpreprocessedTitle = tokenizer.batch_encode_plus(preprocessedTitle)['input_ids']
print(len(encodedpreprocessedTitle))

In [ ]:
embeddigs = model.transformer.wte
print("Shape of embedding matrix : ",embeddigs.weight.shape)
print("Type of embedding matrix : ", type(embeddigs))

In [ ]:
TitleEmbeddingList = []
QIDList = []
for idx, (qid, encodedTitle) in tqdm(enumerate(zip(QID, encodedpreprocessedTitle))):
  if len(encodedTitle) > 0 :
    embeddedTitle = embeddigs(torch.tensor(encodedTitle).to(torch.int64)).mean(axis=0)
    TitleEmbeddingList.append(embeddedTitle)
    QIDList.append(qid)

In [ ]:
numQ = len(TitleEmbeddingList)
embedDim = len(TitleEmbeddingList[0])
print("Number of Titles : ",numQ," and Length of vector of each Title : ",embedDim)

In [ ]:
TitleEmbeddingTensor = torch.cat(TitleEmbeddingList, dim=0)
TitleEmbeddingTensor = torch.reshape(TitleEmbeddingTensor, (numQ, embedDim))
print("Shape of TitleEmbeddingTensor : ",TitleEmbeddingTensor.shape)
print("Type of TitleEmbeddingTensor : ", type(TitleEmbeddingTensor))

In [ ]:
def preprocesstext(text):
  text =  " ".join([word for word in text.split(" ") if word not in stop])
  text = re.sub(r'[^a-zA-Z0-9 ]','',text)
  text = text.lower()
  return text

In [ ]:
def getMostSimilarQuestionsIdx(K, a, b):
  a_norm = a / a.norm(dim=1)[:, None]
  b_norm = b / b.norm(dim=1)[:, None]
  res = torch.mm(a_norm, b_norm.transpose(0,1)).squeeze(0)
  res = res.tolist()
  mostSimIdx = sorted(range(len(res)), key=lambda x: res[x])[-K:]
  return mostSimIdx

In [ ]:
def getMostSimilarQuestions(K, input, QuestionDF, QIDList):
  input = input
  preprocessedinput = preprocesstext(input)
  inputEncoded = tokenizer.batch_encode_plus([preprocessedinput])['input_ids']
  inputEmbedded = embeddigs(torch.tensor(inputEncoded).to(torch.int64)).squeeze(0).mean(axis=0).unsqueeze(0)
  mostSimilarIdx = getMostSimilarQuestionsIdx(K, inputEmbedded, TitleEmbeddingTensor)
  mostSimilarIdx.reverse()
  print("Most similar ",K, " questions : ")
  for idx, simidx in enumerate(mostSimilarIdx):
    IDQ = QuestionDF[QuestionDF['Id']==QIDList[simidx]][['Id','Title']].values
    parentId = IDQ[0][0]
    simQuestion = IDQ[0][1]
    print((idx+1), "Question Id : ", parentId, "Question : ",simQuestion)

In [ ]:
getMostSimilarQuestions(5, "How to MUltiply 2 columns pandas ?", questions ,QIDList)

**Ask reddit troll questions**

In [ ]:
reddit_troll = pd.read_csv("/kaggle/input/askreddit-troll-questions/our_competition_test.csv")
reddit_troll.columns

In [ ]:
troll_questions = list(reddit_troll.question_text)
troll_questions[:10]

In [ ]:
reddit_troll = pd.read_csv("/kaggle/input/askreddit-troll-questions/our_competition_train (1).csv")
reddit_troll.columns

In [ ]:
troll_questions.append(list(reddit_troll.question_text))
troll_questions[:20]

**Amazon questions answers**

In [ ]:
amazon_questions = pd.read_csv("/kaggle/input/amazon-questionanswer-dataset/single_qna.csv")
amazon_questions.head

In [ ]:
amazon_questions.columns

In [ ]:
questions = amazon_questions.Question[:10]

In [ ]:
answers = amazon_questions.Answer[:10]

In [ ]:
qna = list(zip(questions, answers))
qna[:10]

In [ ]:
multi_questions = pd.read_csv("/kaggle/input/amazon-questionanswer-dataset/multi_questions.csv")
multiquestions = multi_questions.QuestionText
len(multiquestions)

In [ ]:
multi_answers = pd.read_csv("/kaggle/input/amazon-questionanswer-dataset/multi_answers.csv")
multianswers = multi_answers.AnswerText
len(multianswers)

***Cleaning Text Data***

In [ ]:
!pip install cleantext

In [ ]:
import cleantext

cleaned_text = []

def clean(textdata, depth):
    if depth == 1:
        for i in textdata:
            cleaned_text.append(cleantext.clean(str(i), extra_spaces=True, lowercase=True, stopwords=True, stemming=True, numbers=True, punct=True, clean_all = True))
        
        return cleaned_text[-10:]
    else:
        for i in textdata:
            for j in i:
                cleaned_text.append(cleantext.clean(str(j), extra_spaces=True, lowercase=True, stopwords=True, stemming=True, numbers=True, punct=True, clean_all = True))
        
        return cleaned_text[-10:]

In [ ]:
clean(dialogs, depth = 1)
del dialogs
clean(emapthetic_dialogs, depth = 1)
del emapthetic_dialogs
clean(qna, depth = 2)
del qna
clean(tc_dialogs, depth = 1)
del tc_dialogs

In [ ]:
clean(jqna, depth = 2)

In [ ]:
del jqna

In [ ]:
clean(troll_questions, depth=1)
del troll_questions

**Getting all questions and answers in 1 variable**

In [ ]:
all_questions = list(dialogs[::2]) + list(mh.Questions) + list(emapthetic_dialogs[::2]) + list(tc_dialogs[::2]) #+ list(jq[' Question']) #+ list(amazon_questions.Question) #+ list(multi_questions.QuestionText)
all_answers = list(dialogs[1::2]) + list(mh.Answers) + list(emapthetic_dialogs[1::2]) + list(tc_dialogs[1::2]) #+ list(jq[' Answer']) #+ list(amazon_questions.Answer) #+ list(multi_answers.AnswerText)

In [ ]:
len(all_questions)

In [ ]:
len(all_answers)

***Data training for chatbot***

In [ ]:
import numpy as np
import tensorflow as tf
import pickle
from tensorflow.keras import layers , activations , models , preprocessing , utils

In [ ]:
tokenizer = preprocessing.text.Tokenizer()
tokenizer.fit_on_texts([str(i) for i in all_questions[:1000]] +  [str(i) for i in all_answers[:1000]])
VOCAB_SIZE = len( tokenizer.word_index )+1
print( 'VOCAB SIZE : {}'.format( VOCAB_SIZE ))

In [ ]:
from gensim.models import Word2Vec
import re

vocab = []
for word in tokenizer.word_index:
    vocab.append( word )

def tokenize( sentences ):
    tokens_list = []
    vocabulary = []
    for sentence in sentences:
        sentence = sentence.lower()
        sentence = re.sub( '[^a-zA-Z]', ' ', sentence )
        tokens = sentence.split()
        vocabulary += tokens
        tokens_list.append( tokens )
    return tokens_list , vocabulary

#p = tokenize( questions + answers )
#model = Word2Vec( p[ 0 ] ) 

#embedding_matrix = np.zeros( ( VOCAB_SIZE , 100 ) )
#for i in range( len( tokenizer.word_index ) ):
    #embedding_matrix[ i ] = model[ vocab[i] ]

# encoder_input_data
tokenized_questions = tokenizer.texts_to_sequences( [str(i) for i in all_questions[:1000]] )
maxlen_questions = max( [ len(x) for x in tokenized_questions ] )
padded_questions = preprocessing.sequence.pad_sequences( tokenized_questions , maxlen=maxlen_questions , padding='post' )
encoder_input_data = np.array( padded_questions )
print( encoder_input_data.shape , maxlen_questions )

# decoder_input_data
tokenized_answers = tokenizer.texts_to_sequences( [str(i) for i in all_answers[:1000]] )
maxlen_answers = max( [ len(x) for x in tokenized_answers ] )
padded_answers = preprocessing.sequence.pad_sequences( tokenized_answers , maxlen=maxlen_answers , padding='post' )
decoder_input_data = np.array( padded_answers )
print( decoder_input_data.shape , maxlen_answers )

# decoder_output_data
tokenized_answers = tokenizer.texts_to_sequences( [str(i) for i in all_answers[:1000]] )
for i in range(len(tokenized_answers)) :
    tokenized_answers[i] = tokenized_answers[i][1:]
padded_answers = preprocessing.sequence.pad_sequences( tokenized_answers , maxlen=maxlen_answers , padding='post' )
onehot_answers = utils.to_categorical( padded_answers , VOCAB_SIZE )
decoder_output_data = np.array( onehot_answers )
print( decoder_output_data.shape )


In [ ]:

encoder_inputs = tf.keras.layers.Input(shape=( maxlen_questions , ))
encoder_embedding = tf.keras.layers.Embedding( VOCAB_SIZE, 200 , mask_zero=True ) (encoder_inputs)
encoder_outputs , state_h , state_c = tf.keras.layers.LSTM( 200 , return_state=True )( encoder_embedding )
encoder_states = [ state_h , state_c ]

decoder_inputs = tf.keras.layers.Input(shape=( maxlen_answers ,  ))
decoder_embedding = tf.keras.layers.Embedding( VOCAB_SIZE, 200 , mask_zero=True) (decoder_inputs)
decoder_lstm = tf.keras.layers.LSTM( 200 , return_state=True , return_sequences=True )
decoder_outputs , _ , _ = decoder_lstm ( decoder_embedding , initial_state=encoder_states )
decoder_dense = tf.keras.layers.Dense( VOCAB_SIZE , activation=tf.keras.activations.softmax ) 
output = decoder_dense ( decoder_outputs )

model = tf.keras.models.Model([encoder_inputs, decoder_inputs], output )
model.compile(optimizer=tf.keras.optimizers.RMSprop(), loss='categorical_crossentropy')

model.summary()

In [ ]:
model.fit([encoder_input_data , decoder_input_data], decoder_output_data, batch_size=50, epochs=150 ) 
model.save( 'model.h5' ) 

In [ ]:
def make_inference_models():
    
    encoder_model = tf.keras.models.Model(encoder_inputs, encoder_states)
    
    decoder_state_input_h = tf.keras.layers.Input(shape=( 200 ,))
    decoder_state_input_c = tf.keras.layers.Input(shape=( 200 ,))
    
    decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
    
    decoder_outputs, state_h, state_c = decoder_lstm(
        decoder_embedding , initial_state=decoder_states_inputs)
    decoder_states = [state_h, state_c]
    decoder_outputs = decoder_dense(decoder_outputs)
    decoder_model = tf.keras.models.Model(
        [decoder_inputs] + decoder_states_inputs,
        [decoder_outputs] + decoder_states)
    
    return encoder_model , decoder_model

In [ ]:

def str_to_tokens( sentence : str ):
    words = sentence.lower().split()
    tokens_list = list()
    for word in words:
        tokens_list.append( tokenizer.word_index[ word ] ) 
    return preprocessing.sequence.pad_sequences( [tokens_list] , maxlen=maxlen_questions , padding='post')

In [ ]:

enc_model , dec_model = make_inference_models()

for _ in range(10):
    states_values = enc_model.predict( str_to_tokens( input( 'Enter question : ' ) ) )
    empty_target_seq = np.zeros( ( 1 , 1 ) )
    empty_target_seq[0, 0] = tokenizer.word_index['start']
    stop_condition = False
    decoded_translation = ''
    while not stop_condition :
        dec_outputs , h , c = dec_model.predict([ empty_target_seq ] + states_values )
        sampled_word_index = np.argmax( dec_outputs[0, -1, :] )
        sampled_word = None
        for word , index in tokenizer.word_index.items() :
            if sampled_word_index == index :
                decoded_translation += ' {}'.format( word )
                sampled_word = word
        
        if sampled_word == 'end' or len(decoded_translation.split()) > maxlen_answers:
            stop_condition = True
            
        empty_target_seq = np.zeros( ( 1 , 1 ) )  
        empty_target_seq[ 0 , 0 ] = sampled_word_index
        states_values = [ h , c ] 

    print( decoded_translation )